# Crypto.Net · 08 — Rust core vs managed C#

*Built by Gravicode Studios, led by Kang Fadhil.*

> **Safe to run.** This notebook only reads public networks and signs offline; it never broadcasts a transaction. Keys created here are demo keys — never send real funds to them.

Open in VS Code with the **Polyglot Notebooks** extension (or Jupyter with .NET Interactive) and run the cells in order. [Versi Bahasa Indonesia](./08-benchmarks.id.ipynb)

What you will do: run the built-in benchmark that times every primitive on both engines. You will also time your own workload with `CryptoNative.UseBackend`, check that both engines produce identical bytes, and see why `Auto` keeps SHA-2 on the platform's hardware-accelerated implementation.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.2.0"

In [ ]:
using System.Diagnostics;
using Crypto.Net.Native;
using Crypto.Net.Extensions;

$"Rust core: {NativeLoader.IsNativeAvailable} ({NativeLoader.RuntimeIdentifier}), default backend {CryptoNative.Backend}"

## The built-in benchmark

`CryptoBenchmark.Run` measures operations per second on each engine. `cnet bench` and the Gallery chart use the same benchmark. `scale` shortens or lengthens each measurement.

In [ ]:
var results = CryptoBenchmark.Run(scale: 0.3);
results.Select(r => new
{
    r.Category,
    r.Name,
    Rust = $"{r.NativeOpsPerSec:N0}/s",
    Managed = $"{r.ManagedOpsPerSec:N0}/s",
    Speedup = r.Speedup > 0 ? $"{r.Speedup:0.0}×" : "n/a",
})

## Your own workload

`CryptoNative.UseBackend` switches engines for the current async flow only, so you can compare inside one process, or pin a request to the managed engine.

In [ ]:
byte[] key = Convert.FromHexString("4c0883a69102937d6231471b5dbb6204fe5129617082792ae468d01a3f362318");
byte[] digest = CryptoNative.Keccak256("hello"u8);

double Time(CryptoBackend backend, int n)
{
    using var _ = CryptoNative.UseBackend(backend);
    var sw = Stopwatch.StartNew();
    for (int i = 0; i < n; i++) CryptoNative.Secp256k1SignRecoverable(key, digest);
    return n / sw.Elapsed.TotalSeconds;
}
Time(CryptoBackend.Native, 50); Time(CryptoBackend.Managed, 50);   // warm up
double native = Time(CryptoBackend.Native, 2000), managed = Time(CryptoBackend.Managed, 2000);
$"ECDSA sign: Rust {native:N0}/s · managed {managed:N0}/s · {native / managed:0.0}× faster"

## Same answer either way

The managed engine is a fallback, not an approximation: CI runs a parity test for every primitive. You can spot-check that here.

In [ ]:
(byte[] Sig, byte Rec) Sign(CryptoBackend b) { using var _ = CryptoNative.UseBackend(b); return CryptoNative.Secp256k1SignRecoverable(key, digest); }
byte[] Blake(CryptoBackend b) { using var _ = CryptoNative.UseBackend(b); return CryptoNative.Blake2b256("hello"u8); }
new
{
    EcdsaIdentical = Sign(CryptoBackend.Native).Sig.SequenceEqual(Sign(CryptoBackend.Managed).Sig),
    Blake2bIdentical = Blake(CryptoBackend.Native).SequenceEqual(Blake(CryptoBackend.Managed)),
}

## Why `Auto` keeps SHA-2 on the platform

SHA-256 and HMAC-SHA512 in .NET use the CPU's SHA extensions through the OS. For small inputs, the cost of calling across the FFI boundary into Rust outweighs what Rust saves. So `Auto` uses Rust for everything except SHA-2, HMAC and PBKDF2.

In [ ]:
double Sha(CryptoBackend b)
{
    using var _ = CryptoNative.UseBackend(b);
    var sw = Stopwatch.StartNew();
    for (int i = 0; i < 200_000; i++) CryptoNative.Sha256(digest);
    return 200_000 / sw.Elapsed.TotalSeconds;
}
Sha(CryptoBackend.Auto);
$"SHA-256 (32 bytes): Auto/platform {Sha(CryptoBackend.Auto):N0}/s · Rust {Sha(CryptoBackend.Native):N0}/s"